#**Universidad Nacional de Loja**#

## **Carrera de Ingeniería en Computación**
# Desarrollador por: Anthony Luzuriaga - anthony.luzuriaga@unl.edu.ec

# Ejecutar todas las instrucciones parar correr la aplicación


Descargue el modelo previamente

Conectar con Google Drive

In [5]:
# Montar Google Drive (solo en Colab)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install gradio

In [7]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 65.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

# **último codigo**

Agregar el modelo a la ruta donde se encuentra el código

In [8]:
import gradio as gr
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO

# Carga el modelo preentrenado de YOLOv8
model_path = "/content/drive/MyDrive/Dataset/best2.pt"
model = YOLO(model_path)

# Función para la detección
def detect_bees(image):
    try:
        img = np.array(image)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        results = model(img)  # Realizar detección
        boxes = results[0].boxes  # Extraer las cajas
        coords = boxes.xyxy.cpu().numpy()
        cls = boxes.cls.cpu().numpy()
        conf = boxes.conf.cpu().numpy()

        detecciones_filtradas = [
            (coords[i], cls[i], conf[i]) for i in range(len(conf)) if conf[i] >= 0.5 and int(cls[i]) == 0
        ]

        for coord, class_id, confidence in detecciones_filtradas:
            x1, y1, x2, y2 = map(int, coord)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, f"{confidence:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        num_abejas = len(detecciones_filtradas)
        mensaje = f"Abejas detectadas: {num_abejas}"

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img), mensaje
    except Exception as e:
        return None, f"Error: {str(e)}"

# Función para limpiar la imagen y el texto de la interfaz
def clear_detection():
    return None, None, ""

# Interfaz de usuario con Gradio
def app():
    with gr.Blocks(css="""
    /* Contenedor principal */
    .gradio-container {
        position: relative;
        z-index: 1;
        background: rgba(255, 255, 255, 0.85);
        border-radius: 10px;
        padding: 20px;
        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        max-width: 1200px;
        margin: auto;
    }

    /* Estilos para títulos llamativos */
    .responsive-title {
        text-align: center;
        color: #FFD700;  /* Amarillo oro */
        font-size: 3.5rem;
        font-weight: bold;
        margin-top: 40px;
        text-shadow: 3px 3px 8px rgba(0, 0, 0, 0.8);
    }

    /* Contenedor de los logos */
    .logo-container {
        position: relative; /* Posiciona el contenedor de manera relativa */
        top: 10px; /* Ajusta la posición desde la parte superior */
        width: 100%; /* Ocupa todo el ancho disponible */
        display: flex; /* Usa flexbox para organizar los logos */
        justify-content: space-between; /* Distribuye los logos equitativamente */
        padding: 0 30px; /* Añade un espacio lateral */
        z-index: 10; /* Asegura que los logos estén sobre otros elementos */
    }

    /* Estilos para imágenes de logos filter: drop-shadow(0px 0px 5px rgba(0, 0, 0, 0.2)) brightness(1.1) contrast(1.2); */
    .logo-container img {
        width: 350px;
        height: 140px;
        object-fit: contain;

        filter: brightness(1.2) contrast(1.3); /* Mejora el brillo y contraste */
        image-rendering: -webkit-optimize-contrast; /* Para navegadores WebKit (Chrome, Safari) */
        image-rendering: crisp-edges; /* Para mejorar los bordes */
        image-rendering: pixelated; /* Útil para imágenes pixeladas */
    }



    /* Botones mejorados */
    .responsive-button {
        background: linear-gradient(45deg, #FFA500, #FF4500);
        color: white;
        border: none;
        padding: 14px 30px;
        font-size: 20px;
        margin: 15px auto;
        border-radius: 8px;
        cursor: pointer;
        font-weight: bold;
        transition: all 0.3s ease-in-out;
    }

    .responsive-button:hover {
        background: linear-gradient(45deg, #FF4500, #FFA500);
        transform: scale(1.05);
    }

    /* Botón de detección específico */
    .detect-button {
        background: linear-gradient(45deg, #4CAF50, #45a049);  /* Verde */
    }

    .detect-button:hover {
        background: linear-gradient(45deg, #45a049, #4CAF50);  /* Verde más oscuro */
    }

    /* Fondo de la página */
    .background-container {
        position: fixed;
        top: 0;
        left: 0;
        width: 100%;
        height: 100%;
        z-index: -1;
    }

    .background-container img {
        width: 100%;
        height: 100%;
        object-fit: cover;
    }

    /* Estilos para la pestaña de detección */
    .detection-title {
        text-align: center;
        color: white;
        font-size: 2rem;
        font-weight: bold;
        text-shadow: 2px 2px 5px rgba(0, 0, 0, 0.7);
    }
    """) as app:

        # Imagen de fondo y logos en HTML
        gr.HTML("""
        <div class="background-container">
            <img src="https://apicolagrijalva.com/wp-content/uploads/2020/11/IMG_20191022_142411-scaled.jpg" alt="Fondo">
        </div>
        <div class="logo-container">
            <img src="https://unl.edu.ec/sites/default/files/inline-images/logogris_0.png" alt="UNL Logo">
            <img src="https://i.postimg.cc/V6m7tf5V/5014942873421983388-1-removebg-preview.png" alt="Carrera Computación">


        </div>
        """)

        with gr.Tabs():
            with gr.TabItem("Inicio"):
                gr.Markdown("""
                <h1 class='responsive-title' style='
                    color: #FFD700;  /* Amarillo dorado */
                    text-shadow: 3px 3px 6px rgba(0, 0, 0, 0.8);
                    font-size: 3rem;
                    text-align: center;
                    padding: 15px;
                '>🐝 Detección de Abejas con YOLOv8 🐝</h1>
                """)


                gr.Markdown("""
                <p style='
                    text-align: center;
                    font-size: 1.4rem;
                    color: #fff;
                    background: rgba(0, 0, 0, 0.7);
                    padding: 15px;
                    border-radius: 10px;
                    max-width: 800px;
                    margin: auto;
                    box-shadow: 0 4px 8px rgba(255, 255, 255, 0.3);
                '>
                🚀 Esta aplicación permite detectar abejas en imágenes utilizando la tecnología de redes neuronales YOLOv8.
                Sube una imagen y el sistema identificará automáticamente la presencia de abejas con un modelo preentrenado.
                </p>
                """)



            with gr.TabItem("Detección"):
                gr.Markdown("""
                  <h2 style='
                      text-align: center;

                      color: #fff;  /* Naranja vibrante */
                      text-shadow: 2px 2px 5px rgba(0, 0, 0, 0.7);
                      font-size: 2.2rem;
                      padding: 10px;
                      border-radius: 10px;
                  '>📷 Sube una imagen para detectar abejas</h2>
                  """)
                input_image = gr.Image(type="pil", label="Sube tu imagen")
                output_image = gr.Image(type="pil", label="Resultado")
                result_text = gr.Textbox(label="Resultados", interactive=False)

                with gr.Row():
                    detect_button = gr.Button("🔍 Detectar", elem_classes=["responsive-button", "detect-button"])
                    clear_button = gr.Button("🗑️ Limpiar", elem_classes=["responsive-button"])

                detect_button.click(
                    detect_bees, inputs=[input_image], outputs=[output_image, result_text]
                )
                clear_button.click(
                    clear_detection, inputs=[], outputs=[input_image, output_image, result_text]
                )

    return app

app().launch()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7cae2378b0354fd9e2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
